In [0]:
# --------------------------------------------------------------------------------------------------------------------------------------------------
# SCRIPT: 4.5_relatorio_mensal_partidos
# LOCAL: 3_gold/src/4_gastos_ceap/
# OBJETIVO: Gerar relatório mensal (10 maiores gastos p/ partido) a partir da tabela gerada no script 3_gold/4_gastos_ceap/4.2_fato_ceap_despesas 
# ENTREGA: Relatório mensal automatizado com top 10 gastos por partido
# --------------------------------------------------------------------------------------------------------------------------------------------------

from pyspark.sql.functions import col, sum, month, year, desc, row_number
from pyspark.sql.window import Window

# Agrupar gastos por Ano, Mês e Partido
df_mensal = spark.table("gold_fato_ceap_despesas") \
    .withColumn("ano", year("data_referencia")) \
    .withColumn("mes", month("data_referencia")) \
    .groupBy("ano", "mes", "partido") \
    .agg(sum("valor_gasto").alias("total_gasto_partido"))

# Criar um ranking (Top 10) para cada mês
window_spec = Window.partitionBy("ano", "mes").orderBy(desc("total_gasto_partido"))

df_top_10 = df_mensal.withColumn("ranking", row_number().over(window_spec)) \
    .filter(col("ranking") <= 10) \
    .orderBy("ano", "mes", "ranking")

# Salvar na Gold
df_top_10.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_relatorio_mensal_ceap")

display(df_top_10)

In [0]:
%sql

CREATE OR REPLACE TABLE gold_relatorio_mensal_ceap AS
WITH resumo_mensal AS (
    SELECT 
        YEAR(data_referencia) as ano,
        MONTH(data_referencia) as mes,
        partido,
        SUM(valor_gasto) as total_gasto_partido
    FROM gold_fato_ceap_despesas
    GROUP BY 1, 2, 3
),
ranking_partidos AS (
    SELECT 
        *,
        ROW_NUMBER() OVER(PARTITION BY ano, mes ORDER BY total_gasto_partido DESC) as ranking
    FROM resumo_mensal
)
SELECT * FROM ranking_partidos 
WHERE ranking <= 10
ORDER BY ano, mes, ranking;


SELECT * FROM gold_relatorio_mensal_ceap;

In [0]:
display(spark.table("gold_relatorio_mensal_ceap"))